# LLM Commentary — Smoke Test

Validates that `llm/commentary.py` produces useful output from `claude-haiku-4-5-20251001` and that **prompt caching is actually firing** (cache_read_input_tokens > 0 on the second call).

Requires `ANTHROPIC_API_KEY` in your `.env` file.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import anthropic
from dotenv import load_dotenv
load_dotenv('../.env')

from utils.config import env, get as cfg
from llm.commentary import _SYSTEM_PROMPT, _get_client

c = cfg()['llm']
print(f"Model: {c['commentary_model']}")
print(f"Max tokens: {c['max_tokens']}")
print(f"API key loaded: {'yes' if env('ANTHROPIC_API_KEY').startswith('sk-') else 'NO — check .env'}")

In [ ]:
# ── Fake signals (mirroring real engine output format) ───────────────────────
FAKE_SIGNALS = [
    {
        "pair": "BTC/USDT:USDT",
        "direction": "long",
        "strategy": "ema_crossover",
        "regime": "trending-up",
        "regime_confidence": 0.72,
        "regime_age_candles": 8,
        "entry_price": 76800.0,
        "suggested_stop_loss": 74950.0,
        "timeframe": "1h",
    },
    {
        "pair": "ETH/USDT:USDT",
        "direction": "short",
        "strategy": "rsi_divergence",
        "regime": "choppy",
        "regime_confidence": 0.61,
        "regime_age_candles": 3,
        "entry_price": 2275.0,
        "suggested_stop_loss": 2340.0,
        "timeframe": "1h",
    },
    {
        "pair": "SOL/USDT:USDT",
        "direction": "long",
        "strategy": "vol_breakout",
        "regime": "high-volatility",
        "regime_confidence": 0.60,
        "regime_age_candles": 5,
        "entry_price": 84.50,
        "suggested_stop_loss": 81.20,
        "timeframe": "1h",
    },
]

FAKE_REGIME_CONTEXT = {
    "BTC/USDT:USDT": {"label": "trending-up",    "confidence": 0.72, "age_candles": 8},
    "ETH/USDT:USDT": {"label": "choppy",         "confidence": 0.61, "age_candles": 3},
    "SOL/USDT:USDT": {"label": "high-volatility", "confidence": 0.60, "age_candles": 5},
}

print(f"{len(FAKE_SIGNALS)} fake signals ready")

In [ ]:
# ── Helper: call the API and show full token usage breakdown ─────────────────
def call_and_show(signal: dict, regime_context: dict, label: str = '') -> str:
    client = _get_client()
    context_block = {
        "signal": {
            k: signal[k] for k in [
                'pair', 'direction', 'strategy', 'regime',
                'regime_confidence', 'regime_age_candles',
                'entry_price', 'suggested_stop_loss', 'timeframe',
            ]
        },
        "current_regimes": regime_context,
    }
    response = client.messages.create(
        model=c['commentary_model'],
        max_tokens=c['max_tokens'],
        system=[
            {
                "type": "text",
                "text": _SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"},
            }
        ],
        messages=[
            {
                "role": "user",
                "content": f"Signal fired:\n{json.dumps(context_block, indent=2)}",
            }
        ],
    )
    text = response.content[0].text.strip()
    u    = response.usage
    cached = getattr(u, 'cache_read_input_tokens', 0) or 0
    created = getattr(u, 'cache_creation_input_tokens', 0) or 0

    tag = f'[{label}] ' if label else ''
    print(f"{tag}{signal['pair']} {signal['direction'].upper()} via {signal['strategy']}")
    print(f"  input={u.input_tokens}  output={u.output_tokens}  "
          f"cache_created={created}  cache_read={cached}")
    cache_status = '✅ CACHE HIT' if cached > 0 else ('📝 cache write' if created > 0 else '❌ no cache')
    print(f"  {cache_status}")
    print(f"\n  Commentary:")
    print(f"  {text}\n")
    return text

In [ ]:
# ── Call 1: BTC long ema_crossover — should write cache ──────────────────────
print("=" * 60)
print("CALL 1 — expect cache_creation_input_tokens > 0")
print("=" * 60)
_ = call_and_show(FAKE_SIGNALS[0], FAKE_REGIME_CONTEXT, 'call 1')

In [ ]:
# ── Call 2: same system prompt, different user message — should HIT cache ────
print("=" * 60)
print("CALL 2 — same system prompt, expect cache_read_input_tokens > 0")
print("=" * 60)
_ = call_and_show(FAKE_SIGNALS[1], FAKE_REGIME_CONTEXT, 'call 2')

In [ ]:
# ── Call 3: third signal — confirm cache keeps hitting ───────────────────────
print("=" * 60)
print("CALL 3 — expect cache_read_input_tokens > 0")
print("=" * 60)
_ = call_and_show(FAKE_SIGNALS[2], FAKE_REGIME_CONTEXT, 'call 3')

In [ ]:
# ── End-to-end: test the actual generate_commentary() function ───────────────
from llm.commentary import generate_commentary

print("Testing generate_commentary() wrapper (returns str or None):\n")
for sig in FAKE_SIGNALS:
    result = generate_commentary(sig, FAKE_REGIME_CONTEXT)
    status = '✅' if result else '❌ returned None'
    preview = result[:120] + '...' if result and len(result) > 120 else result
    print(f"{status} {sig['pair']} {sig['direction']} — {preview}\n")

In [ ]:
# ── Quality checklist — manually verify each commentary ─────────────────────
print("""
Quality checklist (eyeball each commentary above):

  [ ] Names the regime + how many candles it has held
  [ ] States WHY the signal fired (what crossed, where RSI diverged, etc.)
  [ ] Names exactly ONE risk factor
  [ ] No disclaimers / legal language
  [ ] No leverage or position size advice
  [ ] Reads like an analyst, not a chatbot
  [ ] Cache hit on calls 2 and 3 (cache_read_input_tokens > 0)
""")